# Gaussian Naive Bayes 
**IFRI AI Classes**

---

The Gaussian Naive Bayes algorithm is a probabilistic classifier based on Bayes' theorem with strong independence assumptions between features. It is called *naive* because it assumes that all features are conditionally independent given the class label. Despite this simplification, it performs surprisingly well in many real-world applications, especially in text classification and medical diagnosis.


## 1. Key Concepts

Naive Bayes is a generative model that learns the joint probability distribution $P(X, y)$ and uses Bayes' theorem to compute $P(y|X)$ for prediction. Unlike discriminative models, it explicitly models how the data is generated for each class.

The model computes the posterior probability for each class k:

$$P(C_k \mid x) \propto P(C_k) \prod_{j=1}^{n} P(x_j \mid C_k)$$


Where:
- $P(C_k / x)$ is the posterior probability of class $k$ given features $x$.
- $P(C_k)$ is the class prior probability: $n_k / n_{\text{total}}$.
- $P(x_j/C_k)$ is the class-conditional density for feature $j$.
- $n$ is the number of features.

For Gaussian Naive Bayes specifically, each feature $x_j$ is assumed to follow a normal distribution within each class:

$$ P(x_j \mid C_k) = \frac{1}{\sqrt{2\pi \sigma_{kj}^{2}}} \exp\left(-\frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^{2}}\right) $$

Where $mu_{kj}$ and $\sigma_{kj}$ are the mean and standard deviation of feature $j$ for class $k$, estimated via maximum likelihood from the training data.

### Example Illustration
<img src="https://pplx-res.cloudinary.com/image/upload/pplx_search_images/7696d8bb70b76709f33f868df568b14189cab3d3.jpg" 
     width="500" 
     alt="Gaussian Naive Bayes decision boundaries"/>

*Figure 1: Illustration of Gaussian Naive Bayes — each class is modeled by a Gaussian distribution $P(x_j \mid C_k)$ with its own mean $\mu_{kj}$ and its standard deviation $\sigma_{kj}$. The overlap zone represents the region of classification uncertainty.*

Source: *Naive Bayes Classifier, Towards Data Science*

### Numerical Stability

In practice, log probabilities are used instead of raw probabilities to prevent numerical underflow when multiplying many small probabilities:

$$ \log P(C_k \mid x) \propto \log P(C_k) + \sum_{j=1}^{n} \log P(x_j \mid C_k) $$

The final class prediction is the class with the highest log-posterior probability:

$$ \hat{y} = \arg\max_k \left[ \log P(C_k) + \sum_{j=1}^{n} \left(-\frac{1}{2}\log(2\pi\sigma_{kj}^{2}) - \frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^{2}}\right) \right] $$

## 2. Mathematical Foundation

### Class Priors

For each class k, the prior probability is estimated as the proportion of training samples belonging to that class:

$$ P(C_k) = \frac{N_k}{N} $$

Where:
- $N_k$: number of training samples in class $k$
- $N$: total number of training samples

### Gaussian Parameter Estimation

For each class $k$ and each feature $j$:

Mean:

$$ \mu_{kj} = \frac{1}{N_k} \sum_{i:y_i=k} x_{ij} $$

Variance:

$$ \sigma_{kj}^{2} = \frac{1}{N_k} \sum_{i:y_i=k} (x_{ij} - \mu_{kj})^2 $$

### The Independence Assumption

The critical *naive* assumption is:

$$ P(x_1, x_2, \ldots, x_n \mid C_k) = \prod_{j=1}^{n} P(x_j \mid C_k) $$

This is equivalent to assuming zero covariance between features within each class. In practice, this assumption is often violated, but the model remains surprisingly effective because:
- The ranking of posterior probabilities can be correct even if the absolute probabilities are wrong.
- The model has very few parameters, roughly $\mathcal{O}(n_{\text{features}} \times n_{\text{classes}})$, reducing overfitting risk.
- The decision boundaries remain competitive for many real-world problems.

## 3. Pseudo-algorithm


D ← training set of n labeled instances (x_i, y_i) where x_i ∈ ℝ^d, y_i ∈ {1, ..., K}

x ← new unlabeled instance to classify

function GaussianNB_Fit(D)

    for each class k ∈ {1, ..., K} do    
        D_k ← all instances in D where y_i = k
        prior[k] ← |D_k| / |D|
        for each feature j ∈ {1, ..., d} do
            μ[k][j] ← mean of feature j in D_k
            σ²[k][j] ← variance of feature j in D_k + ε
        end for
    end for
    return model = (prior, μ, σ²)
end


function GaussianNB_Predict(x, model)

    (prior, μ, σ²) ← model
    for each class k ∈ {1, ..., K} do
        log_prob[k] ← log(prior[k])
        for each feature j ∈ {1, ..., d} do
            coef ← -0.5 × log(2 × π × σ²[k][j])
            exponent ← -0.5 × (x[j] - μ[k][j])² / σ²[k][j]
            log_prob[k] ← log_prob[k] + coef + exponent
        end for
    end for
    return argmax(log_prob)
end


### Prediction with the model in Python (from the *ifri_mini_ml_lib* implementation):

The `fit` method computes and stores the priors, means, and standard deviations for each class. The `predict` method then uses the `_log_likelihood` internal function to compute the log-posterior for each class and returns the class with the highest score. The standard deviation contains a small epsilon constant $(10^{-9})$ added to avoid numerical division errors. The `predict_proba` method additionally transforms the log-probabilities back into proper probability estimates.


## 4. Implementation

For this implementation, we will use the Iris dataset, which is a commonly used dataset in machine learning. It consists of 150 samples of iris flowers, with 4 features (sepal length, sepal width, petal length, petal width) and 3 classes (setosa, versicolor, virginica).


In [2]:
import pandas as pd
from sklearn.datasets import load_iris
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [3]:
# Split the data into training and testing sets
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)


**With Scikit-learn**


In [4]:
import time
from sklearn.naive_bayes import GaussianNB

start_1 = time.perf_counter()
clf = GaussianNB()
clf.fit(X_train, y_train)
end_1 = time.perf_counter()


**With ifri_mini_ml_lib**


In [5]:
from ifri_mini_ml_lib.classification import NaiveBayes

start_2 = time.perf_counter()
gnb = NaiveBayes()
gnb.fit(X_train.values, y_train.values)
end_2 = time.perf_counter()


In [8]:
# Predict the labels for the test set
y_pred_1 = clf.predict(X_test)
y_pred_2 = gnb.predict(X_test.values)

# Evaluate the accuracy of the model
from ifri_mini_ml_lib.metrics.classification import accuracy, f1_score, recall, precision

accuracy_1 = accuracy(y_test, y_pred_1)
accuracy_2 = accuracy(y_test, y_pred_2)

f1_score_1 = f1_score(y_test, y_pred_1)
f1_score_2 = f1_score(y_test, y_pred_2)

recall_1 = recall(y_test, y_pred_1)
recall_2 = recall(y_test, y_pred_2)

precision_1 = precision(y_test, y_pred_1)
precision_2 = precision(y_test, y_pred_2)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score', 'Recall', 'Precision', 'Time'],
    'Scikit-learn': [accuracy_1, f1_score_1, recall_1, precision_1, (end_1 - start_1)],
    'ifri_mini_ml_lib': [accuracy_2, f1_score_2, recall_2, precision_2, (end_2 - start_2)],
})

results


,Metric,Scikit-learn,ifri_mini_ml_lib
0,Accuracy,1.000000,1.000000
1,F1 Score,1.000000,1.000000
2,Recall,1.000000,1.000000
3,Precision,1.000000,1.000000
4,Time,0.014598,0.001571


Both implementations achieve identical accuracy, but the *ifri_mini_ml_lib* version typically runs faster due to its minimal overhead and direct NumPy operations.


## 5. Interactive Demo


In [15]:
from ipywidgets import interact, FloatSlider, Dropdown
import matplotlib.pyplot as plt
import numpy as np

def plot_naive_bayes_decision(feature_x='sepal length (cm)', feature_y='sepal width (cm)',
                               var_smoothing=1e-9, library='ifri_mini_ml_lib'):
    idx_x = list(iris.feature_names).index(feature_x)
    idx_y = list(iris.feature_names).index(feature_y)

    X_2d_train = X_train_scaled[:, [idx_x, idx_y]]
    X_2d_test  = X_test_scaled[:, [idx_x, idx_y]]

    if library == 'sklearn':
        from sklearn.naive_bayes import GaussianNB
        model = GaussianNB(var_smoothing=var_smoothing)
        model.fit(X_2d_train, y_train)
    else:
        from tiny_ml import GaussianNB as TinyGaussianNB
        model = TinyGaussianNB()
        model.fit(X_2d_train, y_train)

    x_min, x_max = X_2d_test[:, 0].min() - 0.5, X_2d_test[:, 0].max() + 0.5
    y_min, y_max = X_2d_test[:, 1].min() - 0.5, X_2d_test[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    from ifri_mini_ml_lib.metrics.classification import accuracy
    acc = accuracy(y_test, model.predict(X_2d_test))

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='viridis')
    scatter = ax.scatter(X_2d_test[:, 0], X_2d_test[:, 1], c=y_test,
                         cmap='viridis', edgecolors='k', s=40)
    plt.colorbar(scatter, ax=ax, label='Class')
    ax.set_xlabel(feature_x)
    ax.set_ylabel(feature_y)
    ax.set_title(f'Naive Bayes ({library}) — Accuracy: {acc:.3f}')
    plt.tight_layout()
    plt.show()

interact(
    plot_naive_bayes_decision,
    feature_x=Dropdown(options=list(iris.feature_names), value=list(iris.feature_names)[0], description='Feature X:'),
    feature_y=Dropdown(options=list(iris.feature_names), value=list(iris.feature_names)[1], description='Feature Y:'),
    var_smoothing=FloatSlider(min=1e-12, max=1e-6, step=1e-9, value=1e-9, description='Var Smoothing:'),
    library=Dropdown(options=['sklearn', 'ifri_mini_ml_lib'], value='ifri_mini_ml_lib', description='Library:')
);

interactive(children=(Dropdown(description='Feature X:', options=('sepal length (cm)', 'sepal width (cm)', 'pe…

The interactive demo above allows you to visualize the decision boundaries of Gaussian Naive Bayes with different variance smoothing values and across various datasets. You can switch between the Scikit-learn and *ifri_mini_ml_lib* implementations to see how they compare.

The left panel shows the decision regions colored by predicted class. The right panel displays ellipses representing the learned Gaussian distributions: the center of each ellipse is the class mean, and the width and height are proportional to the standard deviation of each feature. Adjusting the variance smoothing parameter controls numerical stability: very small values may lead to overconfident predictions, while larger values tend to produce more regularized, circular Gaussian contours.


## 6. Real-life Applications

Because of its simplicity, speed, and probabilistic nature, Gaussian Naive Bayes is widely used across various domains:

- **Spam detection and email filtering**: Naive Bayes, specifically the multinomial variant, is a classical algorithm for spam filtering. It learns the probability of words appearing in spam versus legitimate emails.
- **Medical diagnosis and clinical decision support**: Researchers use Gaussian NB to predict diseases based on continuous clinical measurements such as blood pressure, cholesterol, and BMI.
- **Real-time anomaly detection in sensor networks**: Because training is extremely fast, Gaussian NB is deployed in industrial IoT for detecting equipment failures.
- **Customer segmentation and behavior prediction**: Marketing teams use Naive Bayes to predict customer purchasing behavior based on demographic and behavioral attributes.
- **Sentiment analysis and NLP**: While Multinomial NB is more common for text, Gaussian NB is used when textual features are encoded as continuous vectors.

Gaussian Naive Bayes excels in situations where:
- Training data is limited.
- Features are roughly normally distributed.
- Fast training and real-time prediction are required.
- Probabilistic outputs with confidence estimates are desired.


## 7. Limitations and Challenges

While Gaussian Naive Bayes is a powerful and efficient algorithm, it has important limitations that must be understood before applying it to real-world problems.

- **Zero-frequency problem**: If a class has zero variance for a particular feature, the Gaussian probability density becomes numerically unstable. This is mitigated by adding a small constant \(\varepsilon\) to the variance.
- **Violation of feature independence**: When features are highly correlated, the model can produce overconfident predictions because it effectively double-counts evidence.
- **Assumption of normality**: If features are highly skewed, multimodal, or categorical, the Gaussian assumption may be inappropriate.
- **Sensitivity to outliers**: Maximum likelihood estimates of mean and variance are sensitive to extreme values.
- **Limited decision boundary shape**: Because each class is modeled by a single Gaussian, the decision boundary is quadratic and may underfit complex patterns.
- **Not competitive on large datasets**: More flexible models such as Random Forests, Gradient Boosting, or Neural Networks often perform better when large amounts of data are available.
- **Poorly calibrated probabilities**: Predicted probabilities are often too extreme and may require calibration techniques such as Platt scaling or isotonic regression.

Overall, Gaussian Naive Bayes is best suited as a fast baseline model or for small-sample scenarios.


## 8. References

- Gaussian Naive Bayes, Scikit-learn Documentation, [https://scikit-learn.org/stable/modules/naive_bayes.html#gaussian-naive-bayes](https://scikit-learn.org/stable/modules/naive_bayes.html#gaussian-naive-bayes)
- Naive Bayes Classifier, Wikipedia, [https://en.wikipedia.org/wiki/Naive_Bayes_classifier](https://en.wikipedia.org/wiki/Naive_Bayes_classifier)
- On Discriminative vs. Generative Classifiers: A Comparison of Logistic Regression and Naive Bayes, Andrew Y. Ng and Michael I. Jordan, NIPS 2002, [https://papers.nips.cc/paper/2001/hash/7e7e69ea3384874304911625ac34321c-Abstract.html](https://papers.nips.cc/paper/2001/hash/7e7e69ea3384874304911625ac34321c-Abstract.html)
- The Optimality of Naive Bayes, Harry Zhang, FLAIRS Conference, 2004, [http://www.cs.unb.ca/~hzhang/publications/FLAIRS04ZhangH.pdf](http://www.cs.unb.ca/~hzhang/publications/FLAIRS04ZhangH.pdf)
- Pattern Recognition and Machine Learning, Christopher M. Bishop, Chapter 8.2.2, Springer, 2006
- Naive Bayes and Text Classification, Sebastian Raschka, [https://sebastianraschka.com/Articles/2014_naive_bayes_1.html](https://sebastianraschka.com/Articles/2014_naive_bayes_1.html)
